# 16. Does a larger calibration set narrow the interval?

Notebook 15 showed that FNR at a 1 percent FPR frozen threshold is identified only up to the gap between two benign order statistics, and that on 399 deepset benigns that gap is wide for the two production detectors. This notebook asks whether the gap closes with more calibration benigns, and separates two effects a referee will want separated:

- **Size effect**: interval width against N, by subsampling within one large pooled benign set (same distribution, more samples).
- **Distribution effect**: interval width across benign sources at a matched N of 399 (same size, different tails).

External benign prompts come from two public instruction datasets (`tatsu-lab/alpaca`, `databricks/databricks-dolly-15k`). They are scored once by each encoder and cached under `data/`, so only the first run needs a GPU and a Hugging Face token for the gated Prompt-Guard checkpoints.

**Outputs**: `reports/calibration_source_interval.csv`, `reports/calibration_width_vs_n.csv`, `reports/calibration_matched_n.csv`, `figures/calibration_width_vs_n.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
os.makedirs('reports', exist_ok=True); os.makedirs('figures', exist_ok=True); os.makedirs('data', exist_ok=True)
print('Session ready.')

## Stage A. Load frames and cached scores (as notebooks 13 to 15)

In [ ]:
import importlib, numpy as np, pandas as pd, os, json, re
import data_loaders, detectors
for m in (data_loaders, detectors): importlib.reload(m)
from data_loaders import (load_deepset, load_bipia_categorized, load_notinject, load_jailbreak,
                          BIPIA_HARMFUL, BIPIA_HIJACK, BIPIA_MALICIOUS)
from detectors import threshold_at_fpr, auroc

deepset = load_deepset()
bip     = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS)
noti    = load_notinject()
jb      = load_jailbreak()
FRAMES  = {'deepset': deepset, 'bipia_cat': bip, 'notinject': noti, 'jailbreak': jb}

TAGS = ['protectai_v2', 'prompt_guard_2', 'prompt_guard_2_22m']
MODEL = {'protectai_v2': 'protectai/deberta-v3-base-prompt-injection-v2',
         'prompt_guard_2': 'meta-llama/Llama-Prompt-Guard-2-86M',
         'prompt_guard_2_22m': 'meta-llama/Llama-Prompt-Guard-2-22M'}
LABEL = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)'}
P, missing = {}, []
for tag in TAGS:
    P[tag] = {}
    for key, f in FRAMES.items():
        fp = f'data/score_{tag}_{key}.npy'
        if not os.path.exists(fp): missing.append(fp); continue
        p = np.load(fp); assert len(p) == len(f), f'{fp}: {len(p)} vs {len(f)} rows'
        P[tag][key] = p
if missing: raise SystemExit('Missing score caches:\n  ' + '\n  '.join(missing))

yd = deepset.label.values; bipben = bip.label.values == 0
H = bip.meta.isin(BIPIA_HARMFUL).values; J = bip.meta.isin(BIPIA_HIJACK).values
# attack score sets (targets) and the cached benign sources per detector
ATK = {tag: {'direct': P[tag]['deepset'][yd == 1], 'indirect_harmful': P[tag]['bipia_cat'][H],
             'indirect_hijack': P[tag]['bipia_cat'][J], 'jailbreak': P[tag]['jailbreak'][jb.label.values == 1]} for tag in TAGS}
BEN = {tag: {'deepset': P[tag]['deepset'][yd == 0], 'jailbreak_benign': P[tag]['jailbreak'][jb.label.values == 0],
             'notinject': P[tag]['notinject']} for tag in TAGS}
print('cached benign sources per detector:', {k: len(v) for k, v in BEN[TAGS[0]].items()})

## Stage B. External benign prompts: load, filter, dedupe, score (cached)

Instruction fields only, 10 to 1000 characters, exact-duplicate removal within and against the three cached benign sets, 3000 sampled per source with a fixed seed. Scoring runs only for cache files that do not exist yet.

In [ ]:
!pip install -q transformers sentencepiece datasets
EXT_N = 3000
EXT = {}
ext_fp = 'data/external_benigns.json'
if os.path.exists(ext_fp):
    EXT = json.load(open(ext_fp)); print('external benign texts loaded from cache:', {k: len(v) for k, v in EXT.items()})
else:
    from datasets import load_dataset
    known = set(str(t).strip() for f in FRAMES.values() for t in f.text.tolist())
    def clean(texts, known):
        out, seen = [], set()
        for t in texts:
            t = str(t if t is not None else '').strip()
            if not (10 <= len(t) <= 1000) or t in seen or t in known: continue
            seen.add(t); out.append(t)
        return out
    alp = load_dataset('tatsu-lab/alpaca', split='train')
    dol = load_dataset('databricks/databricks-dolly-15k', split='train')
    rng = np.random.default_rng(0)
    for name, raw in (('alpaca', alp['instruction']), ('dolly', dol['instruction'])):
        texts = clean(raw, known)                                   # each source is cleaned against everything before it
        idx = rng.choice(len(texts), size=min(EXT_N, len(texts)), replace=False)
        EXT[name] = [texts[i] for i in sorted(idx)]
        known |= set(EXT[name])
    json.dump(EXT, open(ext_fp, 'w'))
    print('external benign texts sampled and cached:', {k: len(v) for k, v in EXT.items()})

need = [(tag, name) for tag in TAGS for name in EXT if not os.path.exists(f'data/score_{tag}_{name}.npy')]
if need:
    from huggingface_hub import login; import getpass
    _t = getpass.getpass('HF token (needed for the gated Prompt-Guard checkpoints): ').strip()
    if _t: login(token=_t)
    from detectors import score_released
    failed = []
    for tag, name in need:
        print('scoring', tag, name, len(EXT[name]))
        try:
            np.save(f'data/score_{tag}_{name}.npy', score_released(EXT[name], MODEL[tag]))
        except Exception as e:
            failed.append(f'{tag}/{name}: {type(e).__name__}: {str(e)[:160]}')
    if failed:
        raise SystemExit('Scoring failed for:\n  ' + '\n  '.join(failed)
                         + '\nThe Prompt-Guard checkpoints are gated: re-run this cell with a valid HF token that has access. Completed scores are cached and will be skipped.')
else:
    print('all external scores cached; no GPU needed')
for tag in TAGS:
    for name in EXT:
        s = np.load(f'data/score_{tag}_{name}.npy'); assert len(s) == len(EXT[name]), f'{tag}/{name}: {len(s)} vs {len(EXT[name])}'
        BEN[tag][name] = s
print('benign sources per detector now:', {k: len(v) for k, v in BEN[TAGS[0]].items()})

## Stage C. Interval helpers

In [ ]:
%%writefile src/calibration_pool.py
"""Identification interval of a frozen 1 percent FPR threshold as a function of the benign calibration pool."""
from __future__ import annotations
import numpy as np
from interval_panel import three_thresholds, fnr, KEYS


def interval_row(pool_ben, atk_sets, fpr_target=0.01):
    """For one benign pool: thresholds, FNR at lo/hi/guar on every attack set, width, and the tail gap."""
    T = three_thresholds(pool_ben, fpr_target)
    out = {"N": int(len(pool_ben)), **{f"t_{k}": T[k] for k in KEYS}, "tail_gap": T["hi"] - T["lo"]}
    for name, pa in atk_sets.items():
        for k in KEYS:
            out[f"FNR_{k}_{name}"] = fnr(pa, T[k])
        out[f"width_{name}"] = out[f"FNR_hi_{name}"] - out[f"FNR_lo_{name}"]
    return out


def width_vs_n(pool_ben, atk_sets, Ns, draws=20, seed=0, fpr_target=0.01):
    """Subsample the pool at each N (without replacement) and average the interval width over draws."""
    rng = np.random.default_rng(seed); rows = []
    for N in Ns:
        N = int(min(N, len(pool_ben)))
        for d in range(draws):
            sub = pool_ben[rng.choice(len(pool_ben), N, replace=False)]
            r = interval_row(sub, atk_sets, fpr_target); r["draw"] = d; rows.append(r)
    return rows


## Stage D. Interval per benign source at full size, and for the pooled set

In [ ]:
import importlib, interval_panel, calibration_pool
importlib.reload(interval_panel); importlib.reload(calibration_pool)
from calibration_pool import interval_row, width_vs_n
from interval_panel import KEYS

SOURCES = ['deepset', 'jailbreak_benign', 'notinject', 'alpaca', 'dolly']
POOLS = {'deepset': ['deepset'], 'jailbreak_benign': ['jailbreak_benign'], 'notinject': ['notinject'],
         'alpaca': ['alpaca'], 'dolly': ['dolly'],
         'pooled_direct(deepset+jb+alpaca+dolly)': ['deepset', 'jailbreak_benign', 'alpaca', 'dolly']}
rows = []
for tag in TAGS:
    for pname, parts in POOLS.items():
        pool = np.concatenate([BEN[tag][s] for s in parts])
        r = interval_row(pool, ATK[tag]); rows.append(dict(detector=LABEL[tag], pool=pname, **r))
SRC = pd.DataFrame(rows); SRC.to_csv('reports/calibration_source_interval.csv', index=False)
pd.set_option('display.width', 250)
cols = ['detector','pool','N','t_lo','t_hi','t_guar','tail_gap','FNR_lo_indirect_hijack','FNR_hi_indirect_hijack','FNR_guar_indirect_hijack','width_indirect_hijack','FNR_lo_indirect_harmful','FNR_hi_indirect_harmful','width_indirect_harmful']
print('=== interval per benign calibration source (full size) and for the pooled set ===')
print(SRC[cols].round(3).to_string(index=False))

## Stage E. Size effect: width against N inside the pooled set

Same distribution, more samples. Twenty random subsamples per N, mean and 90 percent band of the interval width.

In [ ]:
Ns = [100, 200, 400, 800, 1600, 3200, 6400]
rows = []
for tag in TAGS:
    pool = np.concatenate([BEN[tag][s] for s in POOLS['pooled_direct(deepset+jb+alpaca+dolly)']])
    for r in width_vs_n(pool, ATK[tag], Ns, draws=20, seed=0):
        rows.append(dict(detector=LABEL[tag], **r))
WN = pd.DataFrame(rows); WN.to_csv('reports/calibration_width_vs_n.csv', index=False)
agg = WN.groupby(['detector','N']).agg(width_hijack_mean=('width_indirect_hijack','mean'), width_hijack_p5=('width_indirect_hijack', lambda v: np.percentile(v,5)), width_hijack_p95=('width_indirect_hijack', lambda v: np.percentile(v,95)),
                                        width_harmful_mean=('width_indirect_harmful','mean'), FNR_guar_hijack_mean=('FNR_guar_indirect_hijack','mean'), tail_gap_mean=('tail_gap','mean')).reset_index()
print('=== interval width against calibration-set size N (pooled direct benigns) ===')
print(agg.round(3).to_string(index=False))

## Stage F. Distribution effect: every source subsampled to a matched N of 399

In [ ]:
rows = []
for tag in TAGS:
    for s in SOURCES:
        pool = BEN[tag][s]
        for r in width_vs_n(pool, ATK[tag], [399], draws=20, seed=1):
            rows.append(dict(detector=LABEL[tag], source=s, **r))
MN = pd.DataFrame(rows); MN.to_csv('reports/calibration_matched_n.csv', index=False)
agg_mn = MN.groupby(['detector','source']).agg(N=('N','first'), width_hijack=('width_indirect_hijack','mean'), width_harmful=('width_indirect_harmful','mean'),
                                              FNR_guar_hijack=('FNR_guar_indirect_hijack','mean'), tail_gap=('tail_gap','mean')).reset_index()
print('=== interval width per source at matched N = 399 (mean over 20 subsamples) ===')
print(agg_mn.round(3).to_string(index=False))

## Stage G. Figure

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for i, tag in enumerate(TAGS):
    a = agg[agg['detector'] == LABEL[tag]].sort_values('N'); assert len(a), f'no rows for {tag}'
    ax = axes[i]
    ax.plot(a['N'], a['width_hijack_mean'], marker='o', label='hijack, mean width')
    ax.fill_between(a['N'], a['width_hijack_p5'], a['width_hijack_p95'], alpha=.2, label='hijack, 5 to 95 pct over draws')
    ax.plot(a['N'], a['width_harmful_mean'], marker='s', label='harmful, mean width')
    d = SRC[(SRC['detector'] == LABEL[tag]) & (SRC['pool'] == 'deepset')]; assert len(d) == 1
    ax.axhline(float(d['width_indirect_hijack'].iloc[0]), color='k', ls=':', label='published setup (399 deepset), hijack')
    ax.set_xscale('log'); ax.set_xlabel('calibration benigns N (pooled direct)'); ax.set_ylabel('FNR interval width (hi minus lo)')
    ax.set_title(LABEL[tag]); ax.set_ylim(0, max(0.6, float(a['width_hijack_p95'].max()) * 1.1)); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig('figures/calibration_width_vs_n.png', dpi=150); plt.show()

## Stage H. Log and commit

In [ ]:
from reslog import log_result
summary = ('Per-source intervals (full size):\n' + SRC[cols].round(3).to_string(index=False)
           + '\n\nWidth vs N (pooled):\n' + agg.round(3).to_string(index=False)
           + '\n\nMatched N=399 per source:\n' + agg_mn.round(3).to_string(index=False))
log_result('nb16: calibration-set size and source vs interval width', summary, csv_df=SRC, csv_name='calibration_source_interval.csv')
print(summary)

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
                    "nb16: calibration-set size and source effects on the FNR identification interval; add src/calibration_pool.py"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)